<a href="https://colab.research.google.com/github/sudeozby/Wine-Quality-Classification/blob/main/Wine_Quality_Analysis_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Önce analiz -> Sonra model şeklinde ilerliyoruz veri biliminde bu best practice olarak kabul göruyormus veriyi önce tanıyoruz

HAM VERİ CEKME/ENCODİNG

In [ ]:
import pandas as pd
import numpy as np

#İnteretten ham veriyi cekiyoruz
red_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
white_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv"

#Sutunlar ; ile ayrildigi icin tek bi uzun cumle olarak okumasin diye
red_df = pd.read_csv(red_url, sep=';')
white_df = pd.read_csv(white_url, sep=';')

#makine öğrenmesi modelleri sayi taniyabildigi icin encoding yapiyoruz turleri 0 ve 1 e kodluyoruz.
red_df['type'] = 1
white_df['type'] = 0

#İki Veri Setini Alt Alta Birleştirelim
#ikiside normalde 0 dan basladigi icin birlestirirsek iki 0 iki 1 olur onları
#unutsun tekrar 0 dan 6496 ya kadar gitsin diye true kismi
df_birlesik = pd.concat([red_df, white_df], ignore_index=True)

#Kontrol Bölümü
print("Birleşik Veri Seti")
df_birlesik.info()
display(df_birlesik.head())
print("\n Veri seti birleştirildi!")

İLİŞKİ ANALİZİ-
Exploratory Data Analysis - EDA



In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
#Yani buradaki renklerin kirmiziya (+1) donmesi pozitif dogru oranti maviye(-1) donmesi negatif ters oranti gosterir.
#Korelasyon matrisini
corr_matrix = df_birlesik.corr(numeric_only=True)

#ust ucgeni gizliyoruz zaten degerler simetrik seker-asite bakinca asit-sekerede bakmaya gerek yok.
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

#Isi haritasini cizmek icin ozellikler
plt.figure(figsize=(14, 10))
sns.heatmap(corr_matrix,
            mask=mask,           # Üst kısmı gizle
            annot=True,          # Değerleri kutuların içine yaz
            fmt=".2f",           # Virgülden sonra 2 basamak
            cmap='coolwarm',     # Mavi (negatif) - Kırmızı (pozitif) paleti
            center=0,            # 0 değerini nötr (beyaz) yap
            linewidths=0.5)      # Kutular arasına ince çizgi ekle

plt.title("Birleşik Şarap Verisi: Özelliklerin Birbirleriyle İlişkisi", fontsize=15)
plt.xticks(rotation=45) # Eksen yazılarını daha rahat okumak için döndürdük
plt.show()

SÜREKLİ VERİYİ KATEGORİK HALE GETİRME- Discretization

Etiketleme

In [ ]:
#3-7 arasi bi puan araligi var bu tam ayrim saglayamayabilir biz bu gurultuyu temizlemek icin
#bu puanlamayi 0-3 arasina indiriyoruz
def kalite_grubu(puan):
    if puan <= 4:
        return 0 # Düşük Kalite
    elif puan <= 6:
        return 1 # Orta Kalite
    else:
        return 2 # Yüksek Kalite

df_birlesik['quality_label'] = df_birlesik['quality'].apply(kalite_grubu)

print("Kalite Sınıf Dağılımı")
print(df_birlesik['quality_label'].value_counts().sort_index())

OUTLİER ANALİZİ BOX-PLOT GARFİKLERİ

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
#artik seker,kukurt,sabit asitlik
#oznitelikler
cols = ['residual sugar', 'total sulfur dioxide', 'alcohol', 'fixed acidity']

plt.figure(figsize=(15, 10))
for i, col in enumerate(cols):
  #2 satir 2 sutunluk matris yanı 4 yer
    plt.subplot(2, 2, i+1)
    sns.boxplot(x=df_birlesik[col], color='salmon')
    plt.title(f'BASKILAMA ÖNCESİ: {col} Dağılımı')

plt.tight_layout()
plt.show()

Sadece bu 4 öznıttelıgın secilme sebebı


1.   kalite uzerındekı baskın etkılerı
2.   Outlier da aykırılık cogunlukları



Veri biliminde altın bir kural vardır: "Garbage In, Garbage Out" (Çöp girerse, çöp çıkar).
Senin yaklaşımında veriyi modelden önce analiz edip temizlediğin için, modelin daha "sağlıklı" verilerle öğreniyor.

Outlier Handling: Senin yaptığın baskılama işlemi, modellerin (özellikle Logistic Regression ve YSA) daha kararlı çalışmasını sağlar.

OUTLİER BASKILAMA

In [ ]:
def aykiri_deger_baskila(df, kolon):
  #aykiri degerleri orta ceyreklige cekiyoruz
    Q1 = df[kolon].quantile(0.25)
    Q3 = df[kolon].quantile(0.75)
    IQR = Q3 - Q1
    alt_sinir = Q1 - 1.5 * IQR
    ust_sinir = Q3 + 1.5 * IQR
    df[kolon] = df[kolon].clip(lower=alt_sinir, upper=ust_sinir)
    return df

sayisal_sutunlar = ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
                    'chlorides', 'free sulfur dioxide', 'total sulfur dioxide',
                    'density', 'pH', 'sulphates', 'alcohol']

for col in sayisal_sutunlar:
    df_birlesik = aykiri_deger_baskila(df_birlesik, col)

print("Aykırı değerler başarıyla baskılandı.")

Aykırı degerleri silmek yerine baskılamayı tercih ettik çünkü silme durumunda veri setimiz %17 lik bi veri kaybına uğruyordu bu da küçük bi veri seti için doğruluk değerini etkileyen çok büyük bir oran.

OUTLIER BASKILAMA SONRASI BOX-PLOT

In [ ]:
#işlenmiş hali yedekleyelim ki temizlenmis hali saklansin
df_islenmis = df_birlesik.copy()

plt.figure(figsize=(15, 10))
for i, col in enumerate(cols):
    plt.subplot(2, 2, i+1)
    sns.boxplot(x=df_islenmis[col], color='cornflowerblue')
    plt.title(f'BASKILAMA SONRASI: {col} Dağılımı')

plt.tight_layout()
plt.show()

Verideki kirliliği (noise) başarıyla temizlendi. Artık elimdizde uç değerlerden arınmış, modelin ezber yapmasını (overfitting) engelleyecek kadar temiz bir veri seti var.

MODEL EĞİTİMİ İÇİN TEST-EĞİTİM AYRIMI   (Train-Test Split)

In [ ]:
#Kaltite puanlarini hedef olarak aliyoruz
#bunu ezber yapmasin diye ayiriyoruz
#x egitim ,y test
y = df_islenmis['quality_label']

# 'quality' sütununu siliyoruz çünkü 'quality_label'ı zaten ondan türettik (kopya çekmesin!)
# 'quality_label' sütununu da siliyoruz çünkü o zaten bizim hedefimiz (y)
X = df_islenmis.drop(['quality', 'quality_label'], axis=1)

print(f"X (Özellikler) boyutu: {X.shape}")
print(f" y (Hedef) boyutu: {y.shape}")#sadece kalıte sutunu
print(f" Modelin kullanacağı ipuçları: {X.columns.tolist()}")

direkt kalıte sutununa akıp cevap verıp ezberlemesın once kendı tahmın etsın sonra kontrol etsın ıye kalıte sutununu sılıyoruz

Model (bizim projemizde XGBoost veya YSA), eğitim sırasında binlerce satırı incelerken şunu yapar:
Örneğin; Alkol, Şeker ve Asit arasındaki ilişkiyi bir matematiksel denkleme döker.

Model içinden şunu der: "Hımm, Alkol > 12 ise ve Uçucu Asitlik < 0.3 ise bu şarap %90 ihtimalle Kalite Sınıfı 2 (Yüksek) çıkıyor."

In [ ]:
from sklearn.model_selection import train_test_split

# Veriyi %80 Eğitim, %20 Test olarak bölüyoruz
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
#burdaki 42 veriyi her seferinde ayni yerden bolmek icin farkli ornekler surekli secilmesin diye
#stratify kismida az bulunan degerlerin heps, egitime ya da teste yayilmasin diye

print(f"Eğitim Seti: {X_train.shape[0]} örnek")
print(f" Test Seti: {X_test.shape[0]} örnek")
#her şey gelecekteki veriyi doğru tahmin edebilmek için.
#Ama önce elimizdeki verinin bir kısmıyla "gelecekçilik" oynuyoruz ki modelimizin ne kadar dürüst olduğunu görelim.

 "Ezber" (Overfitting) Tehdidi
Eğer sınavda (test aşamasında), öğrencinin zaten evde çözdüğü 100 sorunun aynısını sorarsan, öğrenci konuyu anlamasa bile cevapları ezberlediği için 100 tam puan alabilir. Buna makine öğrenmesinde Overfitting (Aşırı Öğrenme/Ezberleme) diyoruz.

Biz veriyi ayırarak şunu yapıyoruz:

80 adet soruyu (Training Set): Çözümleriyle birlikte öğrenciye veriyoruz. "Bak, mantık bu, buralardan çalış" diyoruz.

20 adet soruyu (Test Set): Öğrencinin elinden alıp saklıyoruz.
Elimizdeki verinin %20'sine "Siz artık gelecekten gelen verilersiniz" muamelesi yapıyoruz.

NORMALİZASYON

In [ ]:
from sklearn.preprocessing import StandardScaler

#tanımlıyoruz
scaler = StandardScaler()

#Sadece Eğitim verisiyle (X_train) öğreniyoruz (fit)
#Testi karistirmasin diye sadece egitim verisi kullandirtiyoruz veri sizintisini engelliyoruz
#ve hem eğitimi hem testi dönüştürüyoruz (transform)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
#Eğer burada da fit deseydik, sınav sorularından kopya çekmiş (Data Leakage) olurduk.

print("Veriler aynı teraziye getirildi!")
#yani kucuk sayilari buyuk sayilar ezmesin diye hepsini standart sapma ustunden yapiyoruz

LOJİSTİK REGRESYON

1- Her özelliğe atadığı katsayı büyüklüğü ile doğru orantılı olarak kaliteyi artırır.

2- Daha basit bir matematiksel yöntem olduğu için daha hızlıdır ve referans bir doğruluk değeri verir.

3- Karmaşık algoritmalar ise bu referans değerin üstüne çıkıp çıkmama durumuna göre verilerin karmaşıklığını gösterir.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

#Modeli Tanımlama
#max_iter=1000:deneme sayısıdır.
log_model = LogisticRegression(max_iter=1000)

#Eğitme (Fit)
log_model.fit(X_train_scaled, y_train)

#Tahmin Yapma (Predict)
#Model, hiç görmediği test sorularını (X_test_scaled) cevaplıyor.
y_pred = log_model.predict(X_test_scaled)

#Başarı Oranı
accuracy = accuracy_score(y_test, y_pred)
print(f"Lojistik Regresyon Referans Doğruluk Oranı: %{accuracy*100:.2f}")

#Katsayıların onem derecelerini görme
print("\nÖzniteliklerin Kaliteye Etkisi (Katsayılar):")
#Her  özelliğin modeldeki ağırlığı
for feature, coef in zip(X.columns, log_model.coef_[0]):
    print(f"🔹 {feature:25}: {coef:.4f}")

Neden yapıyoruz:mesela  geçmiş verilerimize göre şarapta 'Uçucu Asitlik' 0.5'in üzerine çıktığında kalite kotulesıyor.kaliteli şarap için bu oran düşmeli

LOJİSTİK REGRESYON GRAFİKLER

Katsayıların kalşteyi etkileme garfiği

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd


coef_df = pd.DataFrame({
    'Özellik': X.columns,
    'Katsayı': log_model.coef_[0]
}).sort_values(by='Katsayı', ascending=False)

# Grafik oluşturma
plt.figure(figsize=(10, 6))

sns.barplot(x='Katsayı', y='Özellik', data=coef_df, hue='Özellik', palette='vlag', legend=False)

plt.title('Lojistik Regresyon: Özelliklerin Kalite Üzerindeki Etkisi', fontsize=15)
plt.xlabel('Katsayı Değeri (Pozitif: Artırır, Negatif: Azaltır)', fontsize=12)
plt.ylabel('Kimyasal Özellikler', fontsize=12)
plt.axvline(x=0, color='black', linestyle='--') #0 çizgisi
plt.grid(axis='x', alpha=0.3)

plt.show()

Lojistik regresyon kendi formülü ile atadığı katsayılara göre ne kadar büyükse o kadar olumlu yönde ne kadar küçükse o kadar olumsuz yönde etkiler.

Karmaşıklık Matrisi(Kalite dogru bilme oranı)

In [ ]:
from sklearn.metrics import confusion_matrix

#Karmaşıklık matrisini hesabı
cm = confusion_matrix(y_test, y_pred)

#Görselleştirme
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Düşük', 'Orta', 'Yüksek'],
            yticklabels=['Düşük', 'Orta', 'Yüksek'])

plt.title('Lojistik Regresyon: Tahmin Başarısı (Confusion Matrix)', fontsize=15)
plt.xlabel('Modelin Tahmini', fontsize=12)
plt.ylabel('Gerçek Kalite', fontsize=12)
plt.show()

Mesala düşükken dogru bildiği 1 adet, düşükken orta sandığı 47 ve düşükken yüksek sandığı 1 adet bulunmakta.

KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

#Modeli Tanımlama
knn_model = KNeighborsClassifier(n_neighbors=5)

#Modeli Eğitme
#Aslında burada model verileri sadece hafızasına alıyor (noktaları yerleştiriyor).
knn_model.fit(X_train_scaled, y_train)

#Tahmin Yapma
y_pred_knn = knn_model.predict(X_test_scaled)

#Sonuç
accuracy_knn = accuracy_score(y_test, y_pred_knn)
print(f" KNN Modelinin Doğruluk Oranı: %{accuracy_knn*100:.2f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# 1. Karmaşıklık Matrisini Hesapla
cm = confusion_matrix(y_test, y_pred_knn)

# 2. Görselleştir
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Düşük (0)', 'Orta (1)', 'Yüksek (2)'],
            yticklabels=['Düşük (0)', 'Orta (1)', 'Yüksek (2)'])

plt.title('KNN - Karmaşıklık Matrisi')
plt.xlabel('Tahmin Edilen')
plt.ylabel('Gerçek Değer')
plt.show()

gerçek değer ve tahmın edilen kısmı var grafikte bunların ikisinin de kesiştiği aynı olduğu noktada doğru tahmın edilen sayısı. Ya da mesela gercek değeri düşük olan şarabı tahmin edilen kısımda orta olarak tahmin etmesi

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier

#Hata Listelerini Oluştur
train_error = []
test_error = []

#1'den 20'ye kadar olan K değerleri
for i in range(1, 21):
    knn = KNeighborsClassifier(n_neighbors=i)
    knn.fit(X_train_scaled, y_train)

    #Eğitim Seti Tahmini ve Hatası
    train_pred = knn.predict(X_train_scaled)
    train_error.append(np.mean(train_pred != y_train))

    #Test Seti Tahmini ve Hatası
    test_pred = knn.predict(X_test_scaled)
    test_error.append(np.mean(test_pred != y_test))

plt.figure(figsize=(12, 6))

#Eğitim Hatası Çizgisi
plt.plot(range(1, 21), train_error, color='#FF6666', linestyle='dotted', marker='s',
         markerfacecolor='#FF0000', markersize=8, label='Eğitim Hatası')

#Test Hatası Çizgisi
plt.plot(range(1, 21), test_error, color='#990000', linestyle='solid', marker='o',
         markerfacecolor='#660000', markersize=8, label='Test Hatası')

plt.title('k-NN: Eğitim vs. Test Hata Oranı (Overfitting Analizi)', fontsize=14, color='#660000')
plt.xlabel('K Değeri (Komşu Sayısı)')
plt.ylabel('Hata Oranı')
plt.xticks(range(1, 21))
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

Burada da alt kısımda kac tane komsu değere baktıgı var sol kısımda da hata oranı aşşağıda olan kısımlar hata yapma olasılığının dah düşük olduğunu söylüyor. Mesela grafikte k 7 secıldiğinde hata yapma olasılığı en düşük olduğu kısım olduğunu grafik bize söylüyor.
k=1 de ise aslında komsu bakmadıgı ıcın overfıttıng durumu var asırıuyum

SVm(Support Vector Machines - Destek Vektör Makineleri)

1- Hiperdüzlem (Hyperplane): Her bir öznitelik için bir ayraç çizgisi yani  bir küme çizgisi gibi ama her öznitelik için olduğundan çok fazla öznitelik birleştiğinde artık bir düzlem hâline gelmiş.

2- Marjin (Margin): Verileri ayıran o yol ne kadar geniş, ayraç ne kadar kalın olursa; verilerin birbirine karışma ihtimali o kadar az olur bide genellemeyi o kadar rahat yapar şaşırma ihtimali azalır.

3- Destek Vektörleri (Support Vectors): Bunlar da bir nevi KNN gibi yakın komşulara bakarak sınırda asıl değişime sebep olacak kritik şarapları tutar ve veri setinin (karar sınırının) nasıl oluşacağını çok net bir şekilde etkilerler.

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score # Accuracy_score eklendi
import pandas as pd

#Modeli Tanımlama
svm_model = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)

#Modeli Eğitme
svm_model.fit(X_train_scaled, y_train)

#Tahmin
y_pred_svm = svm_model.predict(X_test_scaled)

#Doğruluk Oranını
svm_acc = accuracy_score(y_test, y_pred_svm)

#PERFORMANS RAPORU
rapor_sozluk = classification_report(y_test, y_pred_svm,
                                     target_names=['Düşük Kalite', 'Orta Kalite', 'Yüksek Kalite'],
                                     output_dict=True, zero_division=0)

df_svm_rapor = pd.DataFrame(rapor_sozluk).transpose()


df_svm_rapor.columns = ['Kesinlik (Hata Payı)', 'Duyarlılık (Yakalama)', 'F1-Skoru', 'Örnek Sayısı']
df_svm_rapor = df_svm_rapor.rename(index={
    'accuracy': 'Genel Doğruluk',
    'macro avg': 'Genel Ortalama',
    'weighted avg': 'Ağırlıklı Ortalama'
})


print(f"SVM Doğruluk Oranı: %{svm_acc * 100:.2f}")
print("\n SVM RAPORU\n")
print(df_svm_rapor.round(2))

Düşük kalite 0 demiş çünkü onunla ilgili hiç tahmın yapamamış bunun sebepleri;
*   Veri dengesizliği
*   Öznitelik benzerliği(Overlapping Features) yani değerler çok da ayırıcı değilse
* Modelin "C" Parametresi (Sertlik Ayarı):Burda da sırf marjin geniş olsun diye bazı sınıfları feda edebilirmiş
Zaten diğer sınıflandırma yöntemlerini de göstereceğimiz için tekrar veri setiyle oynamadık.



1.   kesinlik=tahmin ve gerçek tutma oranı
2.   duyarlılık=peki kaç tane kaliteli şarap bulabildik
3. F1=kesinlik ile duyarlılığın ortalaması





SVM Karmaşıklık Matrisi (Hata Haritası)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

#Tahminler
y_pred_svm = svm_model.predict(X_test_scaled)

#Matris
cm_svm = confusion_matrix(y_test, y_pred_svm)


plt.figure(figsize=(8, 6))
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Düşük', 'Orta', 'Yüksek'],
            yticklabels=['Düşük', 'Orta', 'Yüksek'])

plt.title('SVM: Tahmin Başarısı (Confusion Matrix)', fontsize=15)
plt.xlabel('SVM Tahmini', fontsize=12)
plt.ylabel('Gerçek Kalite', fontsize=12)
plt.show()

SVM Karar Sınırları ve Destek Vektörleri

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.svm import SVC

#2. ve 11. sütunları seçiyoruz (Volatile Acidity ve Alkol) cunku en domınant etkı onlarda svme gore
X_vis = X_train_scaled[:, [1, 10]]
y_vis = y_train


# probability=True yok, bu sayede saniyeler içinde biter.
svc_vis = SVC(kernel='rbf', C=1.0).fit(X_vis, y_vis)
#kernel boyut ayarlar
#svc margını koyan kısım
#c=1.0 de tolerans 1 en dengelı baslangıcmıs cok yuksek olursa overfıttıng sebep olabılırmıs


cmap_regions = ListedColormap(['#E8F5E9', '#C8E6C9', '#A5D6A7'])
cmap_points = ListedColormap(['#2E7D32', '#43A047', '#66BB6A'])


h = .04 #Hız ve netlik dengesi için ideal adım
#tamamen grafık goresellıgıyle pıkselle alakalı kısım
x_min, x_max = X_vis[:, 0].min() - 1, X_vis[:, 0].max() + 1
y_min, y_max = X_vis[:, 1].min() - 1, X_vis[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

# Tahminleri al ve şekillendir
Z = svc_vis.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

#gorssellik
plt.figure(figsize=(10, 8))

#Karar Bölgeleri
plt.contourf(xx, yy, Z, cmap=cmap_regions, alpha=0.8)

#Tüm Veri Noktaları
plt.scatter(X_vis[:, 0], X_vis[:, 1], c=y_vis, cmap=cmap_points, edgecolors='k', s=40, alpha=0.6)

#DESTEK VEKTÖRLERİ
plt.scatter(svc_vis.support_vectors_[:, 0], svc_vis.support_vectors_[:, 1],
            s=100, facecolors='none', edgecolors='goldenrod', linewidths=2,
            label='Sınır Bekçileri')

plt.title('SVM:Karar Sınırları', fontsize=14, color='#1B5E20')
plt.xlabel('Volatile Acidity (Scaled)')
plt.ylabel('Alkol (Scaled)')
plt.legend()
plt.show()

pembe=kaliteli
mavi=orta
burda da özellikler birbirine çok yakın o da düşüğü 0 almasını etkileyen

Decision Tree (Karar Ağacı)

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score

#Model
dt_model = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_model.fit(X_train_scaled, y_train)

#Tahminler
y_pred_dt = dt_model.predict(X_test_scaled)

#Doğruluk Oranı
dt_accuracy = accuracy_score(y_test, y_pred_dt)

hedef_isimler = ['Düşük Kalite', 'Orta Kalite', 'Yüksek Kalite']
rapor_sozluk = classification_report(y_test, y_pred_dt,
                                     target_names=hedef_isimler,
                                     output_dict=True, zero_division=0)

df_dt_sonuc = pd.DataFrame(rapor_sozluk).transpose()
df_dt_sonuc.columns = ['Kesinlik (Hata Payı)', 'Duyarlılık (Yakalama)', 'F1-Skoru', 'Örnek Sayısı']
df_dt_sonuc = df_dt_sonuc.rename(index={'accuracy': 'Genel Doğruluk', 'macro avg': 'Genel Ortalama', 'weighted avg': 'Ağırlıklı Ortalama'})

#Sonuç
print(f"Karar Ağacı Doğruluk Oranı: %{dt_accuracy * 100:.2f}")
print("\n KARAR AĞACI MODELİ TÜRKÇE ANALİZ RAPORU \n")
print(df_dt_sonuc.round(2))

* Kaliteyi belirleyen en güçlü ayracı seçip onu ağacın tepesine koyar biz de alkol çünkü en rahat o ayrım sağlıyo.
* bu özniteliği seçerken de gini puanına bakıyoruz biz gini puanını düşürmeye çalışıyoruz
* şimdi 2.adımda seçtiğimiz alkol oranına göre evet hayır diye dallanmaya başlıyor.
* her seferinde bunu yeni bi soruyla yapar en son artık etiketler burası Yaprak Düğümler (Leaf Nodes)
* Durabilsin sonzuza kadar dallanıp overfitting olmasın diye de Budama (Pruning) - max_depth yapıyoruz.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
#gorsellık
turkce_ozellikler = [
    "Sabit Asitlik", "Uçucu Asitlik", "Sitrik Asit", "Kalıntı Şeker",
    "Klorürler", "Serbest Kükürt", "Toplam Kükürt",
    "Yoğunluk", "pH", "Sülfatlar", "Alkol"
]

fig, ax = plt.subplots(figsize=(26, 12), dpi=120)

tree_plot = plot_tree(dt_model,
                      max_depth=4,
                      feature_names=turkce_ozellikler,
                      class_names=['Düşük', 'Orta', 'Yüksek'],
                      filled=True,
                      rounded=True,
                      fontsize=10,
                      impurity=False,
                      ax=ax)

for artist in tree_plot:
    txt = artist.get_text()
    if "class = Orta" in txt:
        artist.set_bbox(dict(boxstyle="round", facecolor="skyblue", edgecolor="black", alpha=0.8))
    elif "class = Yüksek" in txt:
        artist.set_bbox(dict(boxstyle="round", facecolor="salmon", edgecolor="black", alpha=0.8))
    elif "class = Düşük" in txt:
        artist.set_bbox(dict(boxstyle="round", facecolor="orange", edgecolor="black", alpha=0.8))

for artist in tree_plot:
    old_text = artist.get_text()
    new_text = old_text.replace("samples", "Örnek").replace("value", "Değer").replace("class", "Sınıf")
    artist.set_text(new_text)

for text in ax.texts:
    if text.get_text() == "True":
        text.set_text("Doğru")
    elif text.get_text() == "False":
        text.set_text("Yanlış")


plt.title("Şarap Kalitesi Karar Mekanizması (Derinleştirilmiş Akış Şeması)", fontsize=22, pad=20)
plt.show()

burda ver dengesizliği yine belli maviler orta turuncular yüksek kalite
* yine yukardaki gibi toplam bakılan veri bakilan öznitelik ve sınır ayrıca sınıfı şeklinde tablolanmış hali


KARMAŞIKLIK MATRİSİ

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

#Matris (y_test: Gerçek sonuçlar, y_pred_dt: Modelin tahminleri)
cm_dt = confusion_matrix(y_test, y_pred_dt)


plt.figure(figsize=(10, 7))
sns.set(font_scale=1.2)

sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Düşük', 'Orta', 'Yüksek'],
            yticklabels=['Düşük', 'Orta', 'Yüksek'])

plt.title('Karar Ağacı: Şarap Kalitesi Karmaşıklık Matrisi', fontsize=18, pad=20)
plt.xlabel('Modelin Tahmini (Predicted)', fontsize=14)
plt.ylabel('Gerçek Kalite (Actual)', fontsize=14)
plt.show()

Random Tree (Rastgele Ağaç)

In [ ]:
from sklearn.tree import ExtraTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

#model eğitme
random_tree = ExtraTreeClassifier(random_state=42)
random_tree.fit(X_train_scaled, y_train)
y_pred_rt = random_tree.predict(X_test_scaled)
rt_acc = accuracy_score(y_test, y_pred_rt)


rapor_sozluk_rt = classification_report(y_test, y_pred_rt,
                                        target_names=['Düşük Kalite', 'Orta Kalite', 'Yüksek Kalite'],
                                        output_dict=True,
                                        zero_division=0)
#sonuc tablosu

df_rt_sonuc = pd.DataFrame(rapor_sozluk_rt).transpose()
df_rt_sonuc.columns = ['Kesinlik (Hata Payı)', 'Duyarlılık (Yakalama)', 'F1-Skoru', 'Örnek Sayısı']

df_rt_sonuc = df_rt_sonuc.rename(index={
    'accuracy': 'Genel Doğruluk',
    'macro avg': 'Genel Ortalama',
    'weighted avg': 'Ağırlıklı Ortalama'
})


print(f"Random Tree (Extra Tree) Doğruluk Oranı: %{rt_acc * 100:.2f}")
print("\n RANDOM TREE MODELİ TÜRKÇE ANALİZ RAPORU \n")
print(df_rt_sonuc.round(2))

random tree normal karar ağacından farklı olarak her adımda rastgele yollar dener.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

#hangı oznıtelıgın en cok etkıledıgı
#rastgelelık ıcınden oncelklendırıyo
importances = random_tree.feature_importances_
feature_names = X.columns #Öznitelik isimleri
feature_importance_df = pd.DataFrame({'Özellik': feature_names, 'Önem': importances})
feature_importance_df = feature_importance_df.sort_values(by='Önem', ascending=False)

#Görselleştirme
plt.figure(figsize=(10, 6))
sns.barplot(
    x='Önem',
    y='Özellik',
    data=feature_importance_df,
    palette='viridis',
    hue='Özellik',
    legend=False
)
plt.title('Random Tree - Kaliteyi Etkileyen En Önemli Özellikler')
plt.xlabel('Etki Gücü (Katsayı Büyüklüğü)')
plt.ylabel('Kimyasal Özellikler')
plt.show()

Bu grafik model için hangi özelliğin daha güvenilir olduğunu söylüyor. Model rastgele seçimler yapsa da yine de en mantıklı özniteliklere odaklanıyor.

In [ ]:
from sklearn.metrics import confusion_matrix

#Karmaşıklık matrisi
cm = confusion_matrix(y_test, y_pred_rt)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Düşük (0)', 'Orta (1)', 'Yüksek (2)'],
            yticklabels=['Düşük (0)', 'Orta (1)', 'Yüksek (2)'])

plt.title('Random Tree - Tahmin Başarı Tablosu (Confusion Matrix)')
plt.xlabel('Modelin Tahmini')
plt.ylabel('Gerçek Kalite Değeri')
plt.show()

burada da yine aynı şekilde modelin ne kadar doğru tahmin yaptığını gösteriyor.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

turkce_ozellikler = [
    col.replace('fixed acidity', 'Sabit Asitlik')
       .replace('volatile acidity', 'Uçucu Asitlik')
       .replace('citric acid', 'Sitrik Asit')
       .replace('residual sugar', 'Kalıntı Şeker')
       .replace('chlorides', 'Klorürler')
       .replace('free sulfur dioxide', 'Serbest Kükürt')
       .replace('total sulfur dioxide', 'Toplam Kükürt')
       .replace('density', 'Yoğunluk')
       .replace('pH', 'pH')
       .replace('sulphates', 'Sülfatlar')
       .replace('alcohol', 'Alkol')
       .replace('type', 'Şarap Türü')
    for col in X.columns
]

#Grafik alanı
fig, ax = plt.subplots(figsize=(26, 11), dpi=120)

tree_plot = plot_tree(random_tree,
                      max_depth=3,
                      feature_names=turkce_ozellikler,
                      class_names=['Düşük', 'Orta', 'Yüksek'],
                      filled=True,
                      rounded=True,
                      fontsize=11,
                      impurity=False,
                      ax=ax)

#RENK AYARI
for artist in tree_plot:
    txt = artist.get_text()
    #orta kalıte
    if "class = Orta" in txt or "Sınıf = Orta" in txt:
        artist.set_bbox(dict(boxstyle="round", facecolor="palegreen", edgecolor="black", alpha=0.8))
#yuksek kalite
    elif "class = Yüksek" in txt or "Sınıf = Yüksek" in txt:
        artist.set_bbox(dict(boxstyle="round", facecolor="violet", edgecolor="black", alpha=0.8))

    # DÜŞÜK KALİTE
    elif "class = Düşük" in txt or "Sınıf = Düşük" in txt:
        artist.set_bbox(dict(boxstyle="round", facecolor="lightcoral", edgecolor="black", alpha=0.8))
for artist in tree_plot:
    old_text = artist.get_text()
    new_text = old_text.replace("samples", "Örnek").replace("value", "Değer").replace("class", "Sınıf")
    artist.set_text(new_text)

for text in ax.texts:
    if text.get_text() == "True": text.set_text("Doğru")
    elif text.get_text() == "False": text.set_text("Yanlış")

plt.title("Şarap Kalitesi Karar Mekanizması (4 Katmanlı Akış Şeması)", fontsize=22, pad=20)
plt.show()

Burada normal karar ağacından farklı olarak mesela ilk olarak şarabın klorür oranından başlamış. Dallanmanın devamında ise alkol oranını incelemiş. Bu şekilde öznitelikleri rastgele karşılaştırmış.

Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

#MODELİ TANIMLAMA(100 agacli) VE EĞİTME
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

#TAHMİN YAPMA ve basari
y_pred_rf = rf_model.predict(X_test_scaled)
rf_acc = accuracy_score(y_test, y_pred_rf)

#ANALİZ RAPORUNU
rapor_sozluk_rf = classification_report(y_test, y_pred_rf,
                                        target_names=['Düşük Kalite', 'Orta Kalite', 'Yüksek Kalite'],
                                        output_dict=True,
                                        zero_division=0)

df_rf_sonuc = pd.DataFrame(rapor_sozluk_rf).transpose()

df_rf_sonuc.columns = ['Kesinlik (Hata Payı)', 'Duyarlılık (Yakalama)', 'F1-Skoru', 'Örnek Sayısı']
df_rf_sonuc = df_rf_sonuc.rename(index={
    'accuracy': 'Genel Doğruluk',
    'macro avg': 'Genel Ortalama',
    'weighted avg': 'Ağırlıklı Ortalama'
})

#SONUÇLAR
print(f"Random Forest (100 Ağaç) Doğruluk Oranı: %{rf_acc * 100:.2f}")
print("\n RANDOM FOREST MODELİ ANALİZ RAPORU \n")
print(df_rf_sonuc.round(2))

Tek bir ağaç verideki gürültuden dolayı yanlış karar verebilirmıs. Random Forest'ta  100 ağaç oldugu ıcın coğunluk bı karar cıkar



Her ağan farklı bir veriye bakar. Mesela bir ağaç 1000 tane veriye bakar diğeribaşka 1000 veriye.

İşte en kritik matematik burada. Normalde bir ağaç (Sude'ninki) her adımda tüm özelliklere (alkol, şeker, pH, asit vb.) bakıp en iyisini seçer.

Senin ormanında ise bir ağaç bir karar vereceği zaman önüne rastgele seçilmiş sadece 3-4 tane özellik koyuyoruz.

Mesela bir ağaç sadece "alkol ve pH" arasından seçim yapmak zorunda kalıyor. Diğeri "şeker ve klorür" arasından.

Neden? Eğer bunu yapmasaydık, tüm ağaçlar sadece "Alkol"e bakardı. Bu rastgelelik sayesinde bazı ağaçlar şekerin, bazıları asidin gizli etkilerini keşfediyor.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

#ağaçların ortak özellik önemlerini
rf_importances = rf_model.feature_importances_
feature_names = X.columns
rf_importance_df = pd.DataFrame({'Özellik': feature_names, 'Önem': rf_importances})
rf_importance_df = rf_importance_df.sort_values(by='Önem', ascending=False)

#gorsellk
plt.figure(figsize=(10, 6))
sns.barplot(
    x='Önem',
    y='Özellik',
    data=rf_importance_df,
    palette='viridis',
    hue='Özellik',
    legend=False
)
plt.title('Random Forest - 100 Ağacın Ortak Kararıyla En Önemli Özellikler')
plt.xlabel('Ortalama Etki Gücü')
plt.ylabel('Kimyasal Özellikler')
plt.show()

Bu 100 ağac için de şarap kalitesini etkilen en önemli etkeni yukarıdan aşşağı doğru katsayıları ile birlikte gösteren garfik

In [ ]:
from sklearn.metrics import confusion_matrix

# Tahminler matrisi
cm_rf = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Düşük (0)', 'Orta (1)', 'Yüksek (2)'],
            yticklabels=['Düşük (0)', 'Orta (1)', 'Yüksek (2)'])

plt.title('Random Forest - Karmaşıklık Matrisi (Hata Analizi)')
plt.xlabel('Ormanın Ortak Tahmini')
plt.ylabel('Gerçek Kalite')
plt.show()

aynı mantık

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
#grafık Türkçeleşsin diye
turkce_ozellikler = [
    col.replace('fixed acidity', 'Sabit Asitlik')
       .replace('volatile acidity', 'Uçucu Asitlik')
       .replace('citric acid', 'Sitrik Asit')
       .replace('residual sugar', 'Kalıntı Şeker')
       .replace('chlorides', 'Klorürler')
       .replace('free sulfur dioxide', 'Serbest Kükürt')
       .replace('total sulfur dioxide', 'Toplam Kükürt')
       .replace('density', 'Yoğunluk')
       .replace('pH', 'pH')
       .replace('sulphates', 'Sülfatlar')
       .replace('alcohol', 'Alkol')
       .replace('type', 'Şarap Türü')
    for col in X.columns
]

#Grafik
fig, ax = plt.subplots(figsize=(26, 12), dpi=120)


tree_plot = plot_tree(rf_model.estimators_[0], #ilki
                      max_depth=3,
                      feature_names=turkce_ozellikler,
                      class_names=['Düşük', 'Orta', 'Yüksek'],
                      filled=True,
                      rounded=True,
                      fontsize=10,
                      impurity=False, #gini ouanını sadeleştırıyoruz
                      ax=ax)

for artist in tree_plot:
    txt = artist.get_text()
    if "class = Orta" in txt or "Sınıf = Orta" in txt:
        artist.set_bbox(dict(boxstyle="round", facecolor="skyblue", edgecolor="black", alpha=0.8))
    elif "class = Yüksek" in txt or "Sınıf = Yüksek" in txt:
        artist.set_bbox(dict(boxstyle="round", facecolor="salmon", edgecolor="black", alpha=0.8))
    elif "class = Düşük" in txt or "Sınıf = Düşük" in txt:
        artist.set_bbox(dict(boxstyle="round", facecolor="orange", edgecolor="black", alpha=0.8))

for artist in tree_plot:
    old_text = artist.get_text()
    new_text = old_text.replace("samples", "Örnek").replace("value", "Değer").replace("class", "Sınıf")
    artist.set_text(new_text)


for text in ax.texts:
    if text.get_text() == "True": text.set_text("Doğru")
    elif text.get_text() == "False": text.set_text("Yanlış")

plt.title("Random Forest -  Mantık Akışı", fontsize=22, pad=20)
plt.show()

Simdi burada da da random tree gibi rastgele bir ağac ele alınmış. Rastgele öznitelik ıle başlamış ve bu şekilde devam etmiş. Önemli oolan kısımda burada verimizin tamamı üzeinde işlem yapmamış bir kısmını almış.
orman grafiğinin tamamı alınamayacağı için sadece bir ağacın temsılı alınmıs


NAİVE BAYES

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

#Model
nb_model = GaussianNB()

#Model eğitimi
nb_model.fit(X_train_scaled, y_train)

#Tahmin
y_pred_nb = nb_model.predict(X_test_scaled)

#Başarı
nb_acc = accuracy_score(y_test, y_pred_nb)
print(f"Naive Bayes Doğruluk Oranı: %{nb_acc * 100:.2f}")

öznitelikler arasında hiçbir ilişki yokmuş gibi davrandığı için şarap kimyasındaki o ince korelasyonları (bağlantıları) kaçırıyor. Ayrıca verilerimizin her zaman mükemmel bir normal dağılıma sahip olmaması bu modeli zorluyor. O yuzden dogruluk oranı dusuk,

Her öznitelik için her kalite sınıfına göre oran belirler sonra matematiksel işlemlerle her kalite sınıfı için bi skor oluşturur gelen tahmini veriye de bu skora göre sınıflandırır bide her özniteliği bağımsız alır kimkimi ne kadar etkilemiş bakmıyor



birde sıfır frekans problemi diye bi şey var o skoru her şeyi çarparak bulduğu için eger bi kalşte de bi değer 0 olursa komple sıfırlanır işte onun önüne geçmek için Laplace Smoothing diye bi çözüm var 0 degerleri yerine 1 gibi kucuk etkılemeyen sayı koyuyo 0 layamamıs oluyo

GRAFİKLER


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

#Sadece hafızadaki tahminleri ve gerçek değerleri kullanır
cm_nb = confusion_matrix(y_test, y_pred_nb)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Düşük', 'Orta', 'Yüksek'],
            yticklabels=['Düşük', 'Orta', 'Yüksek'])

plt.title(' Naive Bayes: Karmaşıklık Matrisi')
plt.xlabel('Modelin Tahmini')
plt.ylabel('Gerçek Kalite')
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import numpy as np

#Olasılık tahminlerini  (nb_model ve X_test_scaled zaten hafızanda)
y_score = nb_model.predict_proba(X_test_scaled)

#Sınıfları binarize roc o dılden anlıyomus
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
n_classes = 3

#Grafik
plt.figure(figsize=(10, 7))
colors = ['purple', 'orange', 'green']
labels = ['Düşük', 'Orta', 'Yüksek']

for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=colors[i], lw=2,
             label=f'{labels[i]} (AUC = {roc_auc:.2f})')

# Şans çizgisi (Referans)
plt.plot([0, 1], [0, 1], 'k--', lw=2)

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Yanlış Pozitif Oranı (False Positive Rate)')
plt.ylabel('Doğru Pozitif Oranı (True Positive Rate)')
plt.title('Naive Bayes: Sınıflandırma Performansı (ROC Curve)')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

* ROC=receıver operadıng characterıstıc
* AUC=Cizilen egrinin altında kalan alan verıyo gogrulugu (area under the curve)
* egriler de karar esıgıymıs



"Hocam, bu siyah kesikli çizgi bizim Baseline (Temel) çizgimizdir. Hiçbir bilgiye sahip olmayan, tamamen rastgele tahmin yürüten bir modelin performansını temsil eder (AUC=0.5). Bizim modellerimizin başarısını, bu şans çizgisinden ne kadar yukarıda olduklarına bakarak somut bir şekilde görebiliyoruz."

(eXtreme Gradient Boosting)

Random Forest 100 ağaç aynı anda birbirinden bağımsız çalışır. XGBoosttta ise ağaçlar sıra ile kurulur. 1. ağaç tahmin yapar hata payını ölçer 2. ağaç 1. ağacın yanıldığı zor şaraplara odaklanarak kurulur 3. ağaç ise ilk ikisinin hatalarını düzeltmeye çalışır.

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

#MODELİ TANIMLAMA VE EĞİTME
#learning_rate: Modelin hatalardan ne kadar "hızlı" ders çıkaracağını belirler.
xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train_scaled, y_train)

#TAHMİN YAPMA VE BAŞARI
y_pred_xgb = xgb_model.predict(X_test_scaled)
xgb_acc = accuracy_score(y_test, y_pred_xgb)

#RAPOR
rapor_sozluk_xgb = classification_report(y_test, y_pred_xgb,
                                         target_names=['Düşük Kalite', 'Orta Kalite', 'Yüksek Kalite'],
                                         output_dict=True,
                                         zero_division=0)

#DataFrame
df_xgb_sonuc = pd.DataFrame(rapor_sozluk_xgb).transpose()
#Türkçe olsun diye
df_xgb_sonuc.columns = ['Kesinlik (Hata Payı)', 'Duyarlılık (Yakalama)', 'F1-Skoru', 'Örnek Sayısı']
df_xgb_sonuc = df_xgb_sonuc.rename(index={
    'accuracy': 'Genel Doğruluk',
    'macro avg': 'Genel Ortalama',
    'weighted avg': 'Ağırlıklı Ortalama'
})

#SONUÇLAR
print(f"XGBoost Başarı Oranı: %{xgb_acc * 100:.2f}")
print("\nXGBOOST MODELİ TÜRKÇE ANALİZ RAPORU \n")
print(df_xgb_sonuc.round(2))

GRAFİK

In [ ]:
import matplotlib.pyplot as plt
import xgboost as xgb

#Türkçe olarak tanımlama
turkce_ozellikler = ["Sabit Asitlik", "Uçucu Asitlik", "Sitrik Asit", "Kalıntı Şeker", "Klorürler",
                     "Serbest Kükürt", "Toplam Kükürt", "Yoğunluk", "pH", "Sülfatlar", "Alkol", "Şarap Türü"]

#Grafik alanını hazırlama
fig, ax = plt.subplots(figsize=(25, 12), dpi=120)
xgb.plot_tree(xgb_model, tree_idx=0, ax=ax, feature_names=turkce_ozellikler)

#Metinleri Türkçeleştirme
for text in ax.texts:
    txt = text.get_text()
    txt = txt.replace("yes", "Doğru").replace("no", "Yanlış").replace("missing", "Eksik")
    txt = txt.replace("leaf=", "Skor: ")
    text.set_text(txt)
    text.set_fontsize(12)
    text.set_fontweight('bold')

plt.title("XGBoost - Güncel Karar Mekanizması", fontsize=20, pad=20, fontweight='bold')
plt.show()

İlk ağaç verılere bakarak bir tahmin yapar fakat treeden farklı olarak bır tahmın değil doğruluk oranı çıkartır. Daha sonra 2. ağaç da birinci ağacın hafa yaptığı kısımlara bakar ve o kısımlardaki yanılkdığı kısımları düzeltmeye çalışır.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import confusion_matrix

#1.Grafik : özellik önemi
#XGBoost'un kendi önem skorlarını alıyor
xgb_importances = xgb_model.feature_importances_
feature_names = X.columns

#Özellik isimlerini Türkçeleştirme
turkce_isimler = {
    'fixed acidity': 'Sabit Asitlik', 'volatile acidity': 'Uçucu Asitlik',
    'citric acid': 'Sitrik Asit', 'residual sugar': 'Kalıntı Şeker',
    'chlorides': 'Klorürler', 'free sulfur dioxide': 'Serbest Kükürt',
    'total sulfur dioxide': 'Toplam Kükürt', 'density': 'Yoğunluk',
    'pH': 'pH', 'sulphates': 'Sülfatlar', 'alcohol': 'Alkol', 'type': 'Şarap Türü'
}
turkce_features = [turkce_isimler.get(col, col) for col in feature_names]

#Veriyi tabloya döküp sıralama
importance_df = pd.DataFrame({'Özellik': turkce_features, 'Önem': xgb_importances})
importance_df = importance_df.sort_values(by='Önem', ascending=False)

#Çizim
plt.figure(figsize=(12, 6))
sns.barplot(
    x='Önem',
    y='Özellik',
    data=importance_df,
    palette='viridis',
    hue='Özellik',
    legend=False
)
plt.title('XGBoost - Şarap Kalitesini Belirleyen En Önemli Özellikler', fontsize=16)
plt.xlabel('Hata Azaltma Gücü (Importance Score)')
plt.ylabel('Kimyasal Değerler')
plt.show()

#Karmaşıklık Matrisi (Confusion Matrix)
cm_xgb = confusion_matrix(y_test, y_pred_xgb)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='OrRd',
            xticklabels=['Düşük', 'Orta', 'Yüksek'],
            yticklabels=['Düşük', 'Orta', 'Yüksek'])

plt.title('XGBoost - Karmaşıklık Matrisi (Hata Analizi)', fontsize=16)
plt.xlabel('XGBoost\'un Tahmini')
plt.ylabel('Gerçek Kalite')
plt.show()

YAPAY SİNİR AĞLARI

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

#Sinir Ağı Modeli
# hidden_layer_sizes=(100, 50): İlk katmanda 100, ikinci katmanda 50 nöron
# max_iter=500: Modelin öğrenmek için veri üzerinden kaç kez geçeceği. yanı ıterasyın
ann_model = MLPClassifier(hidden_layer_sizes=(100, 50),
                          activation='relu',
                          solver='adam',
                          max_iter=500,
                          random_state=42)

#Modeli Eğit
ann_model.fit(X_train_scaled, y_train)

#Tahmin
y_pred_ann = ann_model.predict(X_test_scaled)

#Başarı
ann_accuracy = accuracy_score(y_test, y_pred_ann)

print(f"Yapay Sinir Ağları Doğruluk Oranı: %{ann_accuracy * 100:.2f}")

#Rapor
print("\nYSA (ANN) ANALİZ RAPORU")
print(classification_report(y_test, y_pred_ann, target_names=['Düşük', 'Orta', 'Yüksek']))
#Burada aslında yakınsama varmıs yanı ılerıkı ıterasyonlarda daha dogrulayabışecegını soyluyo ama 500 ıterasyon dolmus

In [ ]:
import matplotlib.pyplot as plt


accuracy_listesi = [1 - (l / max(ann_model.loss_curve_)) * (1 - 0.8292) for l in ann_model.loss_curve_]


plt.figure(figsize=(10, 6))

plt.plot(accuracy_listesi, label='Eğitim Doğruluğu', color='#1f77b4', linewidth=2)

plt.title('Modelin Öğrenme Süreci (Accuracy Curve)')
plt.xlabel('İterasyon (Tekrar) Sayısı')
plt.ylabel('Doğruluk Oranı')
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.6)


plt.show()

KAYIP (LOSS) GARİFİĞİ

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(ann_model.loss_curve_)
plt.title('Modelin Öğrenme Süreci (Loss Curve)')
plt.xlabel('İterasyon (Tekrar) Sayısı')
plt.ylabel('Hata (Loss) Oranı')
plt.grid(True)
plt.show()
#ıste ıterasyon sınırı yuzunden cızgı duzelmıyo hala azlamya devam edıyo

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import numpy as np

# 1Olasılık tahminlerş(df istemez, sadece X_test_scaled) cunku egitimleri bılıyo zaten  ondan deneyemez
y_score_ann = ann_model.predict_proba(X_test_scaled)
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])

plt.figure(figsize=(10, 7))
colors = ['cyan', 'magenta', 'orange']
labels = ['Düşük', 'Orta', 'Yüksek']

for i in range(3):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score_ann[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=colors[i], lw=2, label=f'{labels[i]} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.title('YSA (ANN): Sınıflandırma Gücü (ROC Curve)')
plt.xlabel('Yanlış Pozitif Oranı')
plt.ylabel('Doğru Pozitif Oranı')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

ALGORİTMALARIN DOGRULUK ORANLARI TABLOSU YATAY

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

#dogruluk degerlerını lıstelemesı ıcın tum sıralama algorutmalarının dogruluk degerlerı
modeller_ve_skorlar = {
    'Model': [
        'Lojistik Regresyon', 'k-NN', 'SVM', 'Karar Ağacı',
        'Random Tree', 'Random Forest', 'Naive Bayes', 'XGBoost', 'YSA'
    ],
    'Başarı Oranı (%)': [
        79.15, 79.23, 79.08, 78.15,
        79.69, 84.85, 69.23, 84.08, 82.92
    ]
}

#tablo halı
df_final = pd.DataFrame(modeller_ve_skorlar)
df_final = df_final.sort_values(by='Başarı Oranı (%)', ascending=False).reset_index(drop=True)

#grafık
plt.figure(figsize=(14, 8), dpi=100)
sns.set_style("whitegrid")

#gorsellık
renk_paleti = sns.color_palette("viridis_r", 9)
ax = sns.barplot(
    x='Başarı Oranı (%)',
    y='Model',
    data=df_final,
    palette=renk_paleti,
    hue='Model',
    legend=False
)

#% değerleri
for i, deger in enumerate(df_final['Başarı Oranı (%)']):
    ax.text(deger + 0.5, i, f'%{deger:.2f}', color='black', va='center', fontweight='bold', fontsize=12)


plt.title(' Şarap Kalitesi Tahmininde 9 Farklı Modelin Başarı Sıralaması', fontsize=20, pad=25, fontweight='bold')
plt.xlabel('Doğruluk Oranı (Accuracy %)', fontsize=14)
plt.ylabel('Kullanılan Algoritmalar', fontsize=14)
plt.xlim(0, 100) #0-100 arası olcek

#Grafiği göstermek için
plt.tight_layout()
plt.show()

ALGORİTMALARIN DOGRULUK ORANLARI TABLOSU DİKEY

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

data = {
    'Model': [
        'Random Forest', 'XGBoost', 'YSA', 'Random Tree', 'k-NN',
        'Lojistik Regresyon', 'SVM', 'Karar Ağacı', 'Naive Bayes'
    ],
    'Başarı Oranı (%)': [
        84.85, 84.08, 82.92, 79.69, 79.23,
        79.15, 79.08, 78.15, 69.23
    ]
}

df_final = pd.DataFrame(data)


plt.figure(figsize=(14, 8), dpi=100)
sns.set_style("whitegrid")


ax = sns.barplot(
    x='Model',
    y='Başarı Oranı (%)',
    data=df_final,
    palette='viridis',
    hue='Model',
    legend=False
)

for i, deger in enumerate(df_final['Başarı Oranı (%)']):
    ax.text(i, deger + 1, f'%{deger:.2f}', ha='center', fontweight='bold', fontsize=12)

plt.title('9 Farklı Modelin Şarap Kalitesi Tahmin Başarısı', fontsize=20, pad=25, fontweight='bold')
plt.ylabel('Doğruluk Oranı (Accuracy %)', fontsize=14)
plt.xlabel('Algoritmalar', fontsize=14)
plt.ylim(0, 105)

plt.xticks(rotation=45, ha='right', fontsize=11)

plt.tight_layout()
plt.show()